# 3D toric code — XZ line, **one cut at a time**

Pick a single $h_z$ cut, look at the $h_x$-sweep observables across the available
$L$, fit a sigmoid per $L$ to get $h_x^c(L)$, then finite-size-extrapolate to
$h_x^c(\infty)$ — the one number you paste into `phase_diagram.ipynb`.

Edit **§1 CONFIG** (`HZ` + a couple of knobs), then Run-All. Companion to the
all-cuts overview `xz_line_L4.ipynb`; the data schema / loaders are shared.


In [ ]:
# ====================== 1 · CONFIG — the one cell to edit ======================
import json, glob, os, re
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

# ---- global plot style (spec) ----
plt.rcParams.update({
    "figure.dpi": 120, "font.size": 11,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.3,
})

ROOT = "/Users/sanzhar123/Desktop/Approximate-Symmetries-TC-main/results"
DIRS = {"fm": f"{ROOT}/xz_line_fm", "s2": f"{ROOT}/xz_line_s2", "energy": f"{ROOT}/xz_line_energy"}

# ---- THE CUT ----
HZ = 0.0              # <-- the single h_z slice to analyze
LS = [4, 5, 6]              # list of L to use, or None = every L present at this cut

# ---- locator & fit ----
LOCATOR  = "B_p"        # observable whose sigmoid inflection defines h_x^c(L).
LOCATOR_FALLBACK = ["O_FM", "S2", "B_p", "M_x"]   # used (in order) if LOCATOR absent at this cut
FIT_FORM = "tanh"      # "tanh" (4-par) or "richards" (5-par, asymmetric)
FM_KURT_MAX = None     # max B3 excess-kurtosis to KEEP an O_FM point; None = no mask (use ALL O_FM points)

# ---- fit window (restrict the hx range the sigmoid sees) ----
HX_WINDOW = None       # (lo, hi) to clip the fit, or None = use the full swept range

# ---- finite-size scaling  h_c(L) = h_c(inf) + b (1/L)^x ----
FSS_FIX_X   = 1.0      # fix the exponent x (needed when only 2 L are available); None = fit x (>=3 L)
X_BATTERY   = [1.0, 1.5, 2.0]   # systematic spread of h_c(inf) over these fixed exponents

# observable registry: name -> (source, value_key, err_key, direction)
# NB: the energy observables carry MCMC error bars too (B_p_err/mx_err/A_v_err from the
# per-run dumps), so B_p/M_x get the same PDG naive/honest treatment as O_FM/S2 in §4.
OBS = {
    "O_FM":   ("fm",     "O",      "Oe",     "up"),
    "S2":     ("s2",     "S2",     "S2e",    "down"),
    "M_x":    ("energy", "mx",     "mx_err", "up"),
    "B_p":    ("energy", "B_p",    "B_p_err","down"),
    "A_v":    ("energy", "A_v",    "A_v_err","down"),
    "Vscore": ("energy", "Vscore", None,     "peak"),
}
LABEL = {"O_FM": r"$O_{FM}^{m}$", "S2": r"$S_2$", "M_x": r"$\langle M_x\rangle$",
         "B_p": r"$\langle B_p\rangle$", "A_v": r"$\langle A_v\rangle$", "Vscore": "Vscore"}
# observables drawn in the curve panel (present ones auto-filter)
SHOW_OBS = ["O_FM", "S2", "M_x", "B_p"]
# color encodes system size: one plasma map reused across ALL figures (dark=small L, orange=large)
_ALL_LS = [4, 5, 6, 7]
COL = dict(zip(_ALL_LS, plt.cm.plasma(np.linspace(0, 0.85, len(_ALL_LS)))))
FIGDIR = os.path.join(os.path.dirname(ROOT), "figures")   # gitignored; savefig target
os.makedirs(FIGDIR, exist_ok=True)
print(f"cut: hz = {HZ}   locator = {LOCATOR}   fit = {FIT_FORM}")


## 2 · Load this cut
Pull every observable for the chosen `HZ` across the available `L`.

In [ ]:
# ====================== 2 · DATA (this cut only) ======================
_LHZ = re.compile(r"_L(\d+)_hz(\d+(?:\.\d+)?)")
def _parse(fn):
    m = _LHZ.search(fn); return (int(m.group(1)), round(float(m.group(2)), 4)) if m else (None, None)

def _load(obs):
    """{L: dict(hx,y,ye,raw)} for OBS[obs] at hz=HZ."""
    src, vk, ek, _ = OBS[obs]
    out = {}
    for jp in sorted(glob.glob(os.path.join(DIRS[src], "*.json"))):
        L, hz = _parse(os.path.basename(jp))
        if L is None or abs(hz - HZ) > 1e-6: continue
        d = json.load(open(jp))
        hx = np.array(d.get("field", []), float)
        if hx.size == 0: continue
        o = np.argsort(hx)
        y = np.array(d.get(vk, []), float)
        ye = np.array(d.get(ek, np.zeros(len(hx))) if ek else np.zeros(len(hx)), float)
        if y.size != hx.size: continue
        out[L] = dict(hx=hx[o], y=y[o], ye=ye[o] if ye.size == hx.size else np.zeros(hx.size), raw=d)
    return out

DATA = {obs: _load(obs) for obs in OBS}
Ls_here = sorted({L for obs in DATA.values() for L in obs})
USE_L = Ls_here if LS is None else [L for L in LS if L in Ls_here]

print(f"hz = {HZ}   L present: {Ls_here}   using: {USE_L}\n")
print(f"{'obs':>7} " + "".join(f"L={L:<10}" for L in USE_L))
for obs in OBS:
    row = f"{obs:>7} "
    for L in USE_L:
        r = DATA[obs].get(L)
        row += (f"{r['hx'].min():.2f}-{r['hx'].max():.2f}({r['hx'].size})".ljust(12)
                if r else "-".ljust(12))
    print(row)


## 3 · Curves vs $h_x$
All observables at this cut, $L$ overlaid. The shaded band is `HX_WINDOW`.

In [ ]:
# ====================== 3 · CURVES vs hx ======================
HREF = {"S2": 3*np.log(2), "M_x": 0.5, "O_FM": 0.5}   # reference lines
panels = [o for o in SHOW_OBS if any(DATA[o].get(L) for L in USE_L)]
n = len(panels); ncol = 2; nrow = (n + 1)//2
fig, ax = plt.subplots(nrow, ncol, figsize=(11, 4.0*nrow), squeeze=False)
for i, (p, a) in enumerate(zip(panels, ax.flat)):
    for L in USE_L:
        r = DATA[p].get(L)
        if not r: continue
        a.plot(r["hx"], r["y"], "o-", ms=4, lw=1.2, color=COL[L], label=f"L={L}")
        if np.any(r["ye"] > 0):                      # quoted-sigma bars, muted
            a.errorbar(r["hx"], r["y"], yerr=r["ye"], fmt="none", ecolor="0.5", capsize=2, zorder=1)
        if p == "B_p":                               # overlay A_v (dashed) on the stabilizer panel
            ra = DATA["A_v"].get(L)
            if ra: a.plot(ra["hx"], ra["y"], "--", lw=1, color=COL[L], alpha=0.6)
    if p in HREF: a.axhline(HREF[p], ls=":", c="0.5", lw=1)
    if HX_WINDOW: a.axvspan(*HX_WINDOW, color="0.85", alpha=0.5, zorder=0)
    a.set(xlabel="$h_x$", ylabel=LABEL[p], title=LABEL[p])
    if i == 0: a.legend(fontsize=8, loc="best")
for a in ax.flat[len(panels):]: a.axis("off")
fig.suptitle(f"XZ line — observables at $h_z={HZ}$", y=1.005); fig.tight_layout(); plt.show()
fig.savefig(f"{FIGDIR}/xz_cut_curves_hz{HZ}.png", dpi=300, bbox_inches="tight")

## 4 · Per-$L$ sigmoid fit → $h_x^c(L)$

Fit both sigmoids per $L$; $h_x^c$ = the fitted inflection (richards: $h_0+w\ln\nu$).

**Honest errors (ported from `vertical_line_hz.ipynb` §6).** The quoted per-point $\sigma_i$
capture only the MCMC sampling variance of a *fixed* trained net; the real scatter is
dominated by seed-to-seed NQS training variance, which shows up as $\chi^2/\nu\gg1$.
Following the PDG prescription for mutually inconsistent points:

1. fit with the quoted errors → $\chi^2/\nu$;
2. inflate $s=\max(1,\sqrt{\chi^2/\nu})$, $\;\sigma_i\to s\,\sigma_i$;
3. bootstrap $h_c(L)$ with the **inflated** errors, re-locating the inflection each resample.

`err_naive` (un-inflated bootstrap) is shown alongside as the "fictional" comparison; the
**honest** error is what feeds the §5 FSS. Treat it as a floor, not the truth. For a locator
with no quoted $\sigma$ (energy observables $\langle B_p\rangle$, $\langle M_x\rangle$) there
is nothing to inflate — the bootstrap scale is the residual RMS instead, and $s$ / `err_naive`
are marked `—`.

In [ ]:
# ====================== 4 · PER-L FIT of the locator ======================
# ---- FIT WINDOW: hx range the sigmoids actually see for THIS cut ----
# Points OUTSIDE this range are EXCLUDED from the fits but still drawn (hollow) in
# the overlay. None -> fall back to §1 HX_WINDOW -> full swept range.
# e.g. at hz=0.1 use only hx in [0.4, 1.0]:   FIT_WINDOW = (0.4, 1.0)
FIT_WINDOW = [0.9, 1.9]
N_BOOT     = 400       # bootstrap resamples per (L, form) for the h_c error
LOCATORS   = ["B_p", "M_x"]  # None -> §1 LOCATOR + fallback, one locator per L (auto).
                       # Or a list e.g. ["O_FM","S2"] to fit & compare several (one panel
                       # each); LOCATORS[0] then drives the §5 FSS.

# Both sigmoids are fit and compared side-by-side; §1 FIT_FORM picks the one that
# drives the §5 FSS.
#   tanh     O = A + B tanh((h-h_c)/w)               -> h_c is a fit parameter (symmetric)
#   richards O = A + (K-A)/[1+exp(-(h-h0)/w)]^nu     -> asymmetric; inflection is NOT h0:
#            h_c = h0 + w ln(nu)  (re-located numerically per bootstrap for safety)
FORMS = ["tanh", "richards"]

def richards(h, A, K, h0, w, nu):
    z = np.clip(-(h - h0)/w, -700, 700); return A + (K - A)/(1.0 + np.exp(z))**nu
def tanh_model(h, A, B, hc, w): return A + B*np.tanh((h - hc)/w)
MODELS = {"richards": richards, "tanh": tanh_model}; NPAR = {"richards": 5, "tanh": 4}

def _inflection(form, p, hspan):
    """h_c of the fitted curve. tanh: exact parameter. richards: analytic
    h0 + w*ln(nu) when it lands inside the data range, else numerically re-located
    (steepest slope) — the safe branch for bootstrap resamples that drive nu to
    extremes and push the analytic inflection off the swept range."""
    if form == "tanh":
        return float(p[2])
    A, K, h0, w, nu = p
    hc = h0 + w*np.log(nu)
    lo, hi = hspan
    if lo <= hc <= hi:
        return float(hc)
    hh = np.linspace(lo, hi, 2001)
    return float(hh[int(np.argmax(np.abs(np.gradient(MODELS["richards"](hh, *p), hh))))])

def _p0_bounds(form, h, y):
    lo, hi = float(h.min()), float(h.max()); hmid, span = 0.5*(lo+hi), (hi-lo) or 1.0
    yl, yr = float(y[0]), float(y[-1])
    if form == "richards":
        return ([yl, yr, hmid, 0.05*span, 1.0],
                ([-np.inf,-np.inf, lo-span, 1e-4, 1e-3], [np.inf, np.inf, hi+span, span, 50.0]))
    return ([0.5*(yl+yr), 0.5*(yr-yl), hmid, 0.05*span],
            ([-np.inf,-np.inf, lo-span, 1e-4], [np.inf, np.inf, hi+span, span]))

def _fm_mask(r):
    if FM_KURT_MAX is None: return np.ones(r["hx"].size, bool)   # kurtosis check disabled
    k = r["raw"].get("b3_max_kurt")
    if k is None: return np.ones(r["hx"].size, bool)
    k = np.array(k, float)[np.argsort(np.array(r["raw"]["field"], float))]
    return k <= FM_KURT_MAX

def _win():
    return FIT_WINDOW if FIT_WINDOW is not None else HX_WINDOW

def _used_mask(obs, r):
    """Points that ENTER the fit: kurtosis-OK (O_FM only) and inside the window."""
    hx = r["hx"]; m = _fm_mask(r) if obs == "O_FM" else np.ones(hx.size, bool)
    w = _win()
    if w: m &= (hx >= w[0]-1e-9) & (hx <= w[1]+1e-9)
    return m

def _pts(obs, L, min_n=1):
    """In-window (hx, y, ye) for one obs/L, or None if fewer than min_n points."""
    r = DATA[obs].get(L)
    if not r: return None
    m = _used_mask(obs, r)
    if m.sum() < min_n: return None
    return r["hx"][m], r["y"][m], r["ye"][m]

def _bootstrap_hc(h, y, e, form, p0, bnds, hspan, B, seed=0):
    """Gaussian bootstrap of the inflection: resample y ~ N(y, e), refit weighted by
    e, re-locate h_c each resample (mirrors vertical_line_hz.ipynb §6)."""
    rng = np.random.default_rng(seed)
    lo, hi = hspan[0]-0.05, hspan[1]+0.05
    kw = dict(sigma=e, absolute_sigma=True) if np.all(e > 0) else {}
    out = []
    for _ in range(B):
        yb = y + rng.normal(0, e)
        try:
            pb, _ = curve_fit(MODELS[form], h, yb, p0=p0, bounds=bnds, maxfev=80000, **kw)
            hc = _inflection(form, pb, hspan)
            if lo < hc < hi: out.append(hc)
        except Exception: pass
    return np.array(out)

def fit_one(h, y, ye, form):
    """Fit + PDG-honest h_c error. Weighted locators (O_FM/S2): s=max(1,sqrt(chi2/nu)),
    err_naive from quoted sigma, err_honest from inflated sigma. Unweighted (energy):
    residual-RMS bootstrap only (nothing to inflate -> s/err_naive = nan)."""
    f = MODELS[form]; p0, bnds = _p0_bounds(form, h, y)
    hspan = (float(h.min()), float(h.max()))
    use_w = np.all(np.isfinite(ye)) and np.all(ye > 0)
    kw = dict(sigma=ye, absolute_sigma=True) if use_w else {}
    popt, _ = curve_fit(f, h, y, p0=p0, bounds=bnds, maxfev=80000, **kw)
    resid = y - f(h, *popt); dof = max(1, len(h) - NPAR[form])
    hc = _inflection(form, popt, hspan)
    if use_w:
        chi2dof = float(np.sum((resid/ye)**2)/dof)
        s = max(1.0, np.sqrt(chi2dof))
        en = _bootstrap_hc(h, y, ye,     form, popt, bnds, hspan, N_BOOT)
        eh = _bootstrap_hc(h, y, s*ye,   form, popt, bnds, hspan, N_BOOT)
        err_naive  = float(en.std()) if len(en) > 5 else np.nan
        err_honest = float(eh.std()) if len(eh) > 5 else np.nan
    else:                                        # no quoted sigma -> residual-RMS bootstrap
        chi2dof = float(np.sum(resid**2)/dof)    # absolute SSR/dof (not a normalized chi2)
        sig_eff = np.full_like(y, np.std(resid) or 1e-6)
        eh = _bootstrap_hc(h, y, sig_eff, form, popt, bnds, hspan, N_BOOT)
        err_naive, s = np.nan, np.nan
        err_honest = float(eh.std()) if len(eh) > 5 else np.nan
    err = err_honest if np.isfinite(err_honest) else err_naive
    return dict(popt=popt, h_c=hc, err=err, err_naive=err_naive, err_honest=err_honest,
                s=s, chi2dof=chi2dof, n=len(h), weighted=use_w,
                inwin=hspan[0]-1e-9 <= hc <= hspan[1]+1e-9)

def _usable(o, L):
    """# of in-window points that would actually ENTER the fit for observable o at L
    (after the window clip and, for O_FM, the FM_KURT_MAX heavy-tail mask)."""
    pts = _pts(o, L, min_n=1)
    return 0 if pts is None else int(pts[0].size)

def _fmt(v, w=8, p=4):
    return (f"{v:>{w}.{p}f}" if (v is not None and np.isfinite(v)) else f"{'—':>{w}}")

def _s_for(obs, L):
    """PDG scale factor for the plotted point bars of locator `obs` at L: from the §1
    FIT_FORM if it fit, else the other form, else 1 (no quoted sigma / unfittable)."""
    for f in [FIT_FORM] + [x for x in FORMS if x != FIT_FORM]:
        fr = FITS.get(obs, {}).get(f, {}).get(L)
        if fr and np.isfinite(fr.get("s", np.nan)):
            return fr["s"]
    return 1.0

# ---- select which locator(s) to fit per L ----
_explicit = bool(LOCATORS)
win_used  = _win()
_need     = NPAR[FIT_FORM] + 1

def _locs_for(L):
    if _explicit:                                   # explicit list -> keep those with enough pts
        return [o for o in LOCATORS if _usable(o, L) >= _need]
    chain = [LOCATOR] + [x for x in LOCATOR_FALLBACK if x != LOCATOR]   # auto: one per L
    hit = next((o for o in chain if _usable(o, L) >= _need), None)
    return [hit] if hit else []

OBSL = {L: _locs_for(L) for L in USE_L}             # OBSL[L] = list of locators fit at L
DISP = list(LOCATORS) if _explicit else sorted({o for L in USE_L for o in OBSL[L]})

print(f"fit window (hx): {win_used if win_used else 'full swept range'}"
      f"    (FIT_FORM = {FIT_FORM} drives §5 FSS; err_honest feeds it)")
print(f"locators: {LOCATORS if _explicit else f'{LOCATOR} + fallback {LOCATOR_FALLBACK} (auto)'}"
      f"   need >= {_need} usable in-window pts for {FIT_FORM}"
      + (f"; O_FM mask {'FM_KURT_MAX='+str(FM_KURT_MAX) if FM_KURT_MAX is not None else 'OFF'}"
         if 'O_FM' in ([LOCATOR] + (list(LOCATORS) if _explicit else [])) else ""))
for L in USE_L:
    picks = OBSL[L]
    if not picks:
        print(f"  L{L}: no usable locator (usable " +
              ", ".join(f"{o}:{_usable(o,L)}" for o in DISP) + ")")
    elif not _explicit and picks[0] != LOCATOR:
        print(f"  L{L}: -> {picks[0]}  ({LOCATOR}={_usable(LOCATOR,L)} usable < {_need})")
    elif _explicit and len(picks) < len(LOCATORS):
        print(f"  L{L}: fitting {picks}  (dropped " +
              ", ".join(f"{o}:{_usable(o,L)}" for o in LOCATORS if o not in picks) + ")")

# fit every (locator, form, L)
FITS = {o: {f: {} for f in FORMS} for o in DISP}
for L in USE_L:
    for obs in OBSL[L]:
        for form in FORMS:
            pts = _pts(obs, L, min_n=NPAR[form]+1)
            FITS[obs][form][L] = None if pts is None else {**fit_one(*pts, form), "obs": obs}

# per-locator table
for obs in DISP:
    print(f"\n=== locator {obs} ===")
    for form in FORMS:
        print(f"[{form}]  {'L':>2} {'npts':>5} {'h_c':>9} {'chi2/dof':>9} "
              f"{'s':>5} {'err_naive':>10} {'err_honest':>11}")
        for L in USE_L:
            fr = FITS[obs][form].get(L)
            if fr is None: continue
            print(f"     {L:>2} {fr['n']:>5} {_fmt(fr['h_c'],9)} {fr['chi2dof']:>9.1f} "
                  f"{_fmt(fr['s'],5,1)} {_fmt(fr['err_naive'],10)} {_fmt(fr['err_honest'],11)}")

# primary locator drives §5 FSS
FIT = {}
for L in USE_L:
    if not OBSL[L]: continue
    prim = LOCATORS[0] if _explicit else OBSL[L][0]
    fr = FITS.get(prim, {}).get(FIT_FORM, {}).get(L)
    if fr is not None: FIT[L] = fr

# overlay: one panel per displayed locator (color = size; tanh solid / richards dashed)
_panel = [o for o in DISP if any(FITS[o][f].get(L) for f in FORMS for L in USE_L)]
if _panel:
    styles = {"tanh": "-", "richards": "--"}
    fig, axes = plt.subplots(1, len(_panel), figsize=(7.3*len(_panel), 5.4), squeeze=False)
    for ax, obs in zip(axes[0], _panel):
        for L in USE_L:
            if obs not in OBSL[L]: continue
            c = COL[L]; r = DATA[obs][L]; hx, y, ye = r["hx"], r["y"], r["ye"]
            used = _used_mask(obs, r); ebar = ye * _s_for(obs, L)
            ax.errorbar(hx[used], y[used], yerr=ebar[used], fmt="o", ms=5, mfc=c, mec="k",
                        mew=0.4, ecolor="0.5", capsize=2, zorder=3)
            if (~used).any():
                ax.errorbar(hx[~used], y[~used], yerr=ebar[~used], fmt="o", ms=5, mfc="none",
                            mec="0.7", ecolor="0.7", capsize=2, zorder=2)
            hw = hx[used]; hh = np.linspace(hw.min(), hw.max(), 400)
            for form in FORMS:
                fr = FITS[obs][form].get(L)
                if fr is None: continue
                ax.plot(hh, MODELS[form](hh, *fr["popt"]), styles[form], color=c, lw=1.4,
                        label=f"L={L} {form}: {fr['h_c']:.3f}±{fr['err']:.3f}")
                ax.axvline(fr["h_c"], ls=styles[form], lw=0.8, color=c, alpha=0.5)
        if win_used: ax.axvspan(*win_used, color="0.85", alpha=0.5, zorder=0)
        ax.set(xlabel="$h_x$", ylabel=f"{obs}")
        ax.set_title(f"{obs} — tanh (—) vs richards (- -)")
        ax.legend(fontsize=7, loc="best")
    fig.suptitle(f"per-L fits at $h_z={HZ}$   (bars s·σ; hollow = excluded)", y=1.02)
    fig.tight_layout(); plt.show()
    # fig.savefig(f"{FIGDIR}/xz_cut_fit_hz{HZ}_{FIT_FORM}.png", dpi=300, bbox_inches="tight")


## 5 · Finite-size scaling → $h_x^c(\infty)$
$h_c(L) = h_c(\infty) + b\,(1/L)^x$, weighted by the **honest** (PDG-inflated) $h_c(L)$
errors from §4. With only 2 $L$ the exponent must be fixed (`FSS_FIX_X`); the systematic
band is the spread over `X_BATTERY`. The last line is what you paste into `phase_diagram.ipynb`.

In [ ]:
# ====================== 5 · FSS -> h_c(inf)  (+ exponent sweep) ======================
# One FSS subplot per locator that §4 fit (e.g. O_FM and S2); single subplot if only one.
H_REF   = None        # known/theoretical h_x^c for this cut, or None (else draws a ref line)
YPAD    = 0.06        # tight ylim padding around the data / intercept spread
X_SWEEP = list(X_BATTERY)      # exponents to overlay as fit variants; also sets the systematic band

def fss(Ls, hc, he, fix_x):
    w_ok = np.all(np.isfinite(he)) and np.all(he > 0)
    kw = dict(sigma=he, absolute_sigma=True) if w_ok else {}
    if fix_x is None:
        f = lambda L, h0, b, x: h0 + b*(1.0/L)**x
        pp, cov = curve_fit(f, Ls, hc, p0=[hc.min(), 0.3, 1.0],
                            bounds=([-np.inf,-np.inf,0.1],[np.inf,np.inf,4.0]), maxfev=80000, **kw)
        return dict(h0=float(pp[0]), h0e=float(np.sqrt(abs(cov[0,0]))), b=float(pp[1]),
                    x=float(pp[2]), xe=float(np.sqrt(abs(cov[2,2]))), w=w_ok)
    f = lambda L, h0, b: h0 + b*(1.0/L)**fix_x
    pp, cov = curve_fit(f, Ls, hc, p0=[hc.min(), 0.3], maxfev=80000, **kw)
    return dict(h0=float(pp[0]), h0e=float(np.sqrt(abs(cov[0,0]))), b=float(pp[1]),
                x=float(fix_x), xe=0.0, w=w_ok)

# color per observable (only the fill of the extrapolated square; finite pts encode size)
OBS_STYLE = {"O_FM": ("o","tab:blue"), "S2": ("s","tab:red"), "B_p": ("D","tab:green"),
             "M_x": ("^","tab:purple"), "A_v": ("v","tab:brown"), "Vscore": ("P","0.4")}

def _fss_on_ax(ax, obs, FITo):
    Ls = np.array(sorted(FITo), float)
    hc = np.array([FITo[int(L)]["h_c"] for L in Ls])
    he = np.array([FITo[int(L)]["err"] for L in Ls])
    free_ok = Ls.size >= 3 and FSS_FIX_X is None
    F = fss(Ls, hc, he, None if free_ok else (FSS_FIX_X if FSS_FIX_X is not None else 1.0))
    sweep_fits = []
    for xx in X_SWEEP:
        try: sweep_fits.append((xx, fss(Ls, hc, he, xx)))
        except Exception: pass
    syst = (0.5*(max(s["h0"] for _,s in sweep_fits) - min(s["h0"] for _,s in sweep_fits))
            if len(sweep_fits) >= 2 else np.nan)
    ocol = OBS_STYLE.get(obs, ("o","tab:blue"))[1]
    grid = np.linspace(0, 1.0/Ls.min(), 200)
    scols = plt.cm.viridis(np.linspace(0.15, 0.85, max(len(sweep_fits), 1)))
    for (xx, Fx), sc in zip(sweep_fits, scols):                 # exponent-sweep variants (diamonds)
        ax.plot(grid, Fx["h0"] + Fx["b"]*grid**xx, "--", color=sc, lw=1.2, alpha=0.9, zorder=2)
        ax.errorbar(0, Fx["h0"], yerr=Fx["h0e"], fmt="D", mfc=sc, mec="k", mew=0.4, ms=7,
                    ecolor="0.5", capsize=3, zorder=4, label=fr"$x={xx}$: {Fx['h0']:.3f}$\pm${Fx['h0e']:.3f}")
    prim = f"x free={F['x']:.2f}" if free_ok else f"x fixed={F['x']:.2f}"
    ax.plot(grid, F["h0"] + F["b"]*grid**F["x"], "-", color=ocol, lw=1.8, zorder=5)  # primary fit
    ax.errorbar(0, F["h0"], yerr=F["h0e"], fmt="s", mfc=ocol, mec="k", mew=0.6, ms=10, ecolor="0.5",
                capsize=3, zorder=6, label=fr"primary ({prim}): $h_c(\infty)={F['h0']:.4f}\pm{F['h0e']:.4f}$")
    for L, y, e in zip(Ls, hc, he):                             # finite-L points (circles, fill=size)
        ax.errorbar(1.0/L, y, yerr=(e if F["w"] else None), fmt="o", mfc=COL[int(L)], mec="k",
                    mew=0.4, ms=7, ecolor="0.5", capsize=2, zorder=3, label=f"L={int(L)}")
    if H_REF is not None:
        ax.axhline(H_REF, color="k", ls=":", lw=1, label=f"ref = {H_REF}")
    allv = list(hc) + [s["h0"] for _,s in sweep_fits] + [F["h0"]] + ([H_REF] if H_REF is not None else [])
    ax.set_ylim(min(allv) - YPAD, max(allv) + YPAD)
    ax.set(xlabel="$1/L$", ylabel="$h_x^c(L)$")
    ax.set_title(f"FSS {obs} at $h_z={HZ}$  ({FIT_FORM})")
    ax.legend(fontsize=7, loc="upper right")
    return dict(F=F, syst=syst, Ls=Ls, hc=hc, free_ok=free_ok)

# locators to plot: those §4 fit with >=2 usable L for FIT_FORM; else fall back to primary FIT
try:
    _locs = [o for o in DISP if sum(FITS[o][FIT_FORM].get(L) is not None for L in USE_L) >= 2]
    _FITL = {o: {L: FITS[o][FIT_FORM][L] for L in USE_L if FITS[o][FIT_FORM].get(L) is not None}
             for o in _locs}
except NameError:
    _FITL = {}
if not _FITL:                                    # fallback: whatever the primary FIT holds
    _FITL = {FIT[next(iter(FIT))]["obs"]: FIT} if FIT else {}
_locs = list(_FITL)
assert _locs, "no locator has >=2 fitted L for FSS"

fig, axes = plt.subplots(1, len(_locs), figsize=(6.6*len(_locs), 5), squeeze=False)
RES = {obs: _fss_on_ax(ax, obs, _FITL[obs]) for ax, obs in zip(axes[0], _locs)}
fig.tight_layout(); plt.show()
# fig.savefig(f"{FIGDIR}/xz_cut_fss_hz{HZ}_{FIT_FORM}.png", dpi=300, bbox_inches="tight")

PRIMARY = LOCATORS[0] if (bool(LOCATORS) and LOCATORS[0] in _locs) else _locs[0]
print(f"cut hz = {HZ}   (fit form = {FIT_FORM})")
for obs in _locs:
    R = RES[obs]; F = R["F"]
    xtag = "x free" if R["free_ok"] else f"x={F['x']:.2f}"
    tag = "  <- primary (phase_diagram)" if obs == PRIMARY else ""
    print(f"  [{obs}] h_c(L): " + ", ".join(f"L{int(L)}={h:.4f}" for L,h in zip(R['Ls'],R['hc']))
          + f"  ->  h_c(inf) = {F['h0']:.4f} +/- {F['h0e']:.4f} (stat)"
          + (f" +/- {R['syst']:.4f} (syst)" if np.isfinite(R['syst']) else "") + f"  [{xtag}]{tag}")
    error = np.sqrt((F['h0e'])**2 + (R['syst'])**2)
    print(f"Error added in quadrature is {error:.4f}")
Fp, Lp = RES[PRIMARY]["F"], RES[PRIMARY]["Ls"]
print(f"\n  # paste into phase_diagram.ipynb (primary = {PRIMARY}):")
print(f"  ({HZ}, {Fp['h0']:.4f}, {Fp['h0e']:.4f}),   # h_z, h_x^c, err   [L={[int(x) for x in Lp]}, {FIT_FORM}, {PRIMARY}]")
# print(f'Error added in quadrature is {}')